# Respiratory AI Training on Colab

This notebook trains the final balanced clip dataset on a Colab GPU.

Expected Google Drive files:
- `MyDrive/respiratory_ai_rebuild_colab_project.zip`
- `MyDrive/dataset_final.zip`


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import shutil
import zipfile

PROJECT_ZIP = Path('/content/drive/MyDrive/respiratory_ai_rebuild_colab_project.zip')
DATASET_ZIP = Path('/content/drive/MyDrive/dataset_final.zip')
WORKDIR = Path('/content/respiratory_ai_rebuild')
DATASET_EXTRACTED = Path('/content/dataset_final')

if WORKDIR.exists():
    shutil.rmtree(WORKDIR)
if DATASET_EXTRACTED.exists():
    shutil.rmtree(DATASET_EXTRACTED)
WORKDIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(PROJECT_ZIP, 'r') as zf:
    zf.extractall(WORKDIR)
with zipfile.ZipFile(DATASET_ZIP, 'r') as zf:
    zf.extractall('/content')

assert (WORKDIR / 'src').exists(), f'Missing extracted project files under {WORKDIR}'
assert DATASET_EXTRACTED.exists(), f'Missing extracted dataset at {DATASET_EXTRACTED}'

target_dataset = WORKDIR / 'dataset_final'
if target_dataset.exists() or target_dataset.is_symlink():
    target_dataset.unlink() if target_dataset.is_symlink() else shutil.rmtree(target_dataset)
target_dataset.symlink_to(DATASET_EXTRACTED)

print('Project ready at', WORKDIR)
print('Dataset linked from', DATASET_EXTRACTED)


In [ ]:
%cd /content/respiratory_ai_rebuild
!python -V
!nvidia-smi
!pip install -q PyYAML==6.0.2 librosa==0.10.2.post1 soundfile==0.12.1 Flask==3.0.3 Flask-Cors==4.0.1 seaborn==0.13.2


In [ ]:
%cd /content/respiratory_ai_rebuild
!PYTHONPATH=src python -m resp_ai.models.train --config configs/train_final.yaml


In [ ]:
%cd /content/respiratory_ai_rebuild
!PYTHONPATH=src python -m resp_ai.models.evaluate --config configs/train_final.yaml --model-path models_final/latest/best_model.keras --split test


In [ ]:
from pathlib import Path
import shutil

results_dir = Path('/content/drive/MyDrive/respiratory_ai_rebuild_results')
results_dir.mkdir(parents=True, exist_ok=True)
shutil.copytree('/content/respiratory_ai_rebuild/models_final', results_dir / 'models_final', dirs_exist_ok=True)
print('Saved models to', results_dir / 'models_final')
